In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.under_sampling import NearMiss
import os

def variance(data, threshold=(0.1)):

    sel = VarianceThreshold(threshold)  # removing features of low variance ( rest are functions for creating datasets of different random states and processing descriptors in the same way as previous
    sel.fit_transform(data)
    
    return data[data.columns[sel.get_support(indices=True)]]

def findCorrelation(corr, cutoff=0.9, exact=None):
    
    def _findCorrelation_fast(corr, avg, cutoff):

        combsAboveCutoff = corr.where(lambda x: (np.tril(x)==0) & (x > cutoff)).stack().index

        rowsToCheck = combsAboveCutoff.get_level_values(0)
        colsToCheck = combsAboveCutoff.get_level_values(1)

        msk = avg[colsToCheck] > avg[rowsToCheck].values
        deletecol = pd.unique(np.r_[colsToCheck[msk], rowsToCheck[~msk]]).tolist()

        return deletecol


    def _findCorrelation_exact(corr, avg, cutoff):

        x = corr.loc[(*[avg.sort_values(ascending=False).index]*2,)]

        if (x.dtypes.values[:, None] == ['int64', 'int32', 'int16', 'int8']).any():
            x = x.astype(float)

        x.values[(*[np.arange(len(x))]*2,)] = np.nan

        deletecol = []
        for ix, i in enumerate(x.columns[:-1]):
            for j in x.columns[ix+1:]:
                if x.loc[i, j] > cutoff:
                    if x[i].mean() > x[j].mean():
                        deletecol.append(i)
                        x.loc[i] = x[i] = np.nan
                    else:
                        deletecol.append(j)
                        x.loc[j] = x[j] = np.nan
        return deletecol

    
    if not np.allclose(corr, corr.T) or any(corr.columns!=corr.index):
        raise ValueError("correlation matrix is not symmetric.")
        
    acorr = corr.abs()
    avg = acorr.mean()
        
    if exact or exact is None and corr.shape[1]<100:
        return _findCorrelation_exact(acorr, avg, cutoff)
    else:
        return _findCorrelation_fast(acorr, avg, cutoff)

def preprocess_multiple_states(input_csv_path, output_folder, state_list):
    # Make base output folder once
    os.makedirs(output_folder, exist_ok=True)

    main = pd.read_csv(input_csv_path).drop(columns=['Unnamed: 0'])
    main_y = main['Class']

    for state in state_list:
        print(f"\nProcessing seed: {state}")
        full_train, full_test = train_test_split(main, test_size=0.2, stratify=main_y, random_state=state)

        # train set processing
        df_str = full_train['Structure']  # smiles
        df_y = full_train['Class']  # classif

        train = full_train.drop(columns=['Structure', 'Class']) # variance
        train_var = variance(train)

        train_corr = train_var.corr()
        hc = findCorrelation(train_corr, cutoff=0.9, exact=True) # correlation
        train_var_corr = train_var.drop(columns=hc)

        RobScaler = RobustScaler(unit_variance=True) # scaling
        RobScaler.fit(train_var_corr)

        train_var_corr_scal = RobScaler.transform(train_var_corr)
        train_var_corr_scal = pd.DataFrame(train_var_corr_scal, columns=train_var_corr.columns)

        str_clf = pd.concat([df_str.reset_index(drop=True), df_y.reset_index(drop=True)], axis=1)
        df2 = pd.concat([str_clf, train_var_corr_scal], axis=1)

        # downsampling
        nearMiss = NearMiss(sampling_strategy='majority', version=2, n_neighbors=5, n_jobs=-1)

        df2_x = df2.drop(columns=['Structure'])
        df2_y = df2['Class']
        df2_str = pd.DataFrame(df2['Structure'])

        df2_x_res, df2_y_res = nearMiss.fit_resample(df2_x, df2_y)

        index = nearMiss.sample_indices_
        df2_smiles = df2_str.iloc[index].reset_index(drop=True)

        df_res = pd.concat([df2_smiles, df2_x_res], axis=1)

        # test set processing
        df_test = full_test[df_res.columns]

        test_smiles = df_test['Structure']
        test_y = df_test['Class']

        df_test_scal = RobScaler.transform(df_test.drop(columns=['Structure', 'Class'])) # test set scaling
        df_test_scal = pd.DataFrame(df_test_scal, columns=df_test.drop(columns=['Structure', 'Class']).columns)

        test_smi_class = pd.concat([test_smiles.reset_index(drop=True), test_y.reset_index(drop=True)], axis=1)
        test_set = pd.concat([test_smi_class, df_test_scal], axis=1)

        # Create per-seed subfolder
        seed_folder = os.path.join(output_folder, f'seed_{state}')
        os.makedirs(seed_folder, exist_ok=True)

        # Save to seed-specific subfolder
        train_path = os.path.join(seed_folder, f'train.csv')
        test_path = os.path.join(seed_folder, f'test.csv')

        df_res.to_csv(train_path, index=False)
        test_set.to_csv(test_path, index=False)

        print(f"Saved for seed {state}:")
        print(f"   {train_path}")
        print(f"   {test_path}")


In [30]:
import random

random.seed(1)

states = random.sample(range(0, 10001), 100)

preprocess_multiple_states(
    input_csv_path=r'M:\ML_scripts\mordred_mold2_rdkit_descr_tob_5889.csv',
    output_folder=r'M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets',
    state_list=states)


Processing seed: 2201
Saved for seed 2201:
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_2201\train.csv
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_2201\test.csv

Processing seed: 9325
Saved for seed 9325:
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_9325\train.csv
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_9325\test.csv

Processing seed: 1033
Saved for seed 1033:
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_1033\train.csv
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_1033\test.csv

Processing seed: 4179
Saved for seed 4179:
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_4179\train.csv
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_4179\test.csv

Processing seed: 1931
Saved for seed 1931:
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_1931\train.csv
   M:\ML_scripts\MODEL DATA\RS_datasets\RS_100_sets\seed_1931\test.csv

Processing seed: 8117
Saved for seed 8117:
   M:\ML_scripts\MODE